[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/10_embeddings_contextuales_oraciones/10_proyecto_embeddings_contextuales_oraciones.ipynb)
# Proyecto 10 · ¿Qué vector por documento? Pooling y post-procesado en el retrieval de Políglota

| | |
|---|---|
| **Nivel** | 🟡 Intermedio |
| **Duración** | ≈ 2 h |
| **Cómputo** | CPU en `FAST_DEV_RUN` (≈ 4 min, la mayor parte construyendo los encoders de fallback); con modelos reales, T4 recomendada |
| **Requisitos** | Lección 10 |

## Contexto de negocio
En **Políglota Labs** aún no hay embedder propio (llega en el módulo 11). Producto quiere una primera versión de búsqueda semántica **ya**, con encoders preentrenados.
Tu tarea: decidir, **con datos y sin hacer trampas con el test**, qué combinación *encoder × pooling × post-procesado* usar, y documentar cuánto se gana o pierde frente a la configuración «por defecto» (media, sin post-procesado) y frente a BM25 (módulo 00).

**Conexión.** Usa el corpus y las métricas del módulo 00; la configuración ganadora es el *baseline* denso que tu embedder entrenado deberá batir en el módulo 11.

## Dataset
Corpus Políglota (`carga_corpus`): Wikipedia ES/EN + código (CodeSearchNet o stdlib) o, sin red, sintético con el mismo esquema. Las consultas se parten **50/50 en validación y test** de forma determinista: eliges en validación y reportas en test.

## Entregables y rúbrica
Umbrales calibrados con la solución de referencia en la ejecución que pudo correrse al construir el curso (fallback: mini-BERT crudo + mini-SBERT, corpus sintético, semilla 42). Con modelos reales, ejecuta la solución y usa sus números menos el margen.

| # | Entregable | Criterio |
|---|---|---|
| R1 | `media_mascara`, `max_mascara`, `cls_pool`, `ultimo_token` | pasan los tests unitarios de la celda de comprobación |
| R2 | `ajusta_whitening(X, k)` | con $k=d/2$, la covarianza de $(X-\mu)W$ difiere de $I_k$ en < 1e-3 (máx. absoluto) |
| R3 | `centra_por_idioma` | la media por idioma de los documentos transformados es ≈ 0 (< 1e-6) |
| R4 | Tabla en **validación** con ≥ 2 encoders × 4 poolings × 4 post-procesados (ninguno, centrado, whitening-k, centrado por idioma) | 32+ filas |
| R5 | Configuración elegida en validación, medida en **test** | nDCG@10 test ≥ el de «media sin post-procesado» del mismo encoder − 0.03 (referencia, fallback reproducible: la elegida, mini-SBERT · media · *whitening*, ganó +0.12 en validación (0.499 vs 0.380) y +0.17 en test (0.523 vs 0.352). Con versiones anteriores del fallback, no deterministas, la misma elección quedó entre −0.017 y −0.003 en test: el umbral no exige que el post-procesado gane, solo que la selección en validación no te haga perder más de 0.03; con ~55 consultas por partición la selección es ruidosa, y eso es parte de lo que debes analizar) |
| R6 | Desglose por idioma (es/en/code) de la configuración elegida y comparación con BM25 | tabla impresa |

In [ ]:
!pip install -q transformers sentence-transformers tokenizers datasets scipy scikit-learn matplotlib
import copy, json, math, os, random, re, time, warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

warnings.filterwarnings("ignore")
import transformers
transformers.logging.set_verbosity_error(); transformers.utils.logging.disable_progress_bar()
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
FAST_DEV_RUN = True   # True: corpus pequeño y modelos diminutos de fallback en minutos de CPU
PAL = {"azul": "#1F6FEB", "naranja": "#F28E2B", "verde": "#2CA02C", "gris": "#6B7280", "morado": "#9467BD"}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
print(f"device={device} · FAST_DEV_RUN={FAST_DEV_RUN} · torch {torch.__version__}")

### Utilidades (corpus, métricas, BM25 canónicos y encoders). No hace falta modificarlas.

In [ ]:
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
# ── Métricas de retrieval canónicas de Políglota (módulo 00) ──────────────────
import math

import numpy as np


def _dedup(ranked_ids):
    """Quita duplicados conservando la primera aparición (un doc repetido NO puntúa dos veces)."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def dcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """DCG con ganancia lineal (como trec_eval / pytrec_eval, que usa BEIR/MTEB) y descuento 1/log2(rango+1)."""
    ranked = _dedup(ranked_ids)[:k]
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)


def ndcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """nDCG@k. rels: doc_id -> relevancia (>0 relevante). Sin relevantes -> 0.0 (evaluate() las excluye)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg


def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Fracción de los documentos relevantes que aparecen en el top-k."""
    relevant = {d for d, r in rels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant)


def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k (0 si no hay ninguno)."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def rank_from_scores(scores, doc_ids, k: int | None = None):
    """Ordena por score desc. Empates -> orden original del corpus (determinista). Los empates son una fuente
    silenciosa de varianza: con scores iguales, el orden de entrada decidiría el resultado."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    order = order[:k] if k else order
    return [doc_ids[i] for i in order]


def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    """run: {qid: [doc_id ordenados]}. Promedia solo sobre queries con algún relevante (convención trec_eval)."""
    n, tot = 0, {"ndcg": 0.0, "recall": 0.0, "mrr": 0.0}
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        ranked = run.get(qid, [])
        tot["ndcg"] += ndcg_at_k(ranked, rels, k)
        tot["recall"] += recall_at_k(ranked, rels, k)
        tot["mrr"] += mrr_at_k(ranked, rels, k)
        n += 1
    out = {m: (v / n if n else 0.0) for m, v in tot.items()}
    out.update(n_queries=n, n_skipped=len(qrels) - n)
    return out

In [ ]:
# ── Tokenización simple + BM25 propio (módulo 00) ─────────────────────────────
import math
import re
import unicodedata
from collections import Counter, defaultdict

import numpy as np

_WORD_RE = re.compile(r"\w+", re.UNICODE)


def fold_accents(text: str) -> str:
    """'canción' -> 'cancion' (NFD y descartar marcas combinantes Mn)."""
    nfd = unicodedata.normalize("NFD", text)
    return unicodedata.normalize("NFC", "".join(c for c in nfd if unicodedata.category(c) != "Mn"))


def tokenize_simple(text: str, accents: str = "keep") -> list[str]:
    """Minúsculas + palabras \\w+. accents='fold' quita tildes. (Esto NO es el 'token' de un LLM: es un token léxico.)"""
    text = unicodedata.normalize("NFC", text).lower()
    if accents == "fold":
        text = fold_accents(text)
    return _WORD_RE.findall(text)


class BM25:
    """BM25 con índice invertido. idf='lucene': ln(1+(N-n+.5)/(n+.5)) (siempre >= 0).
    idf='okapi': ln((N-n+.5)/(n+.5)) con suelo epsilon*idf_medio (como rank_bm25.BM25Okapi)."""

    def __init__(self, k1: float = 1.2, b: float = 0.75, idf: str = "lucene", epsilon: float = 0.25):
        self.k1, self.b, self.idf_kind, self.epsilon = k1, b, idf, epsilon

    def fit(self, docs_tokens: list[list[str]], doc_ids: list[str]) -> "BM25":
        self.doc_ids = list(doc_ids)
        self.N = len(docs_tokens)
        self.dl = np.array([len(t) for t in docs_tokens], dtype=float)
        self.avgdl = float(self.dl.mean()) if self.N else 0.0
        self.postings = defaultdict(list)  # término -> [(índice_doc, tf)]
        for i, toks in enumerate(docs_tokens):
            for term, tf in Counter(toks).items():
                self.postings[term].append((i, tf))
        raw = {t: len(p) for t, p in self.postings.items()}
        if self.idf_kind == "lucene":
            self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in raw.items()}
        else:
            self.idf = {t: math.log(self.N - n + 0.5) - math.log(n + 0.5) for t, n in raw.items()}
            floor = self.epsilon * sum(self.idf.values()) / len(self.idf)
            self.idf = {t: (v if v >= 0 else floor) for t, v in self.idf.items()}
        return self

    def scores(self, q_tokens: list[str]) -> np.ndarray:
        s = np.zeros(self.N)
        for term in q_tokens:  # los términos repetidos de la query suman varias veces (igual que rank_bm25)
            for i, tf in self.postings.get(term, ()):
                norm = self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl)
                s[i] += self.idf[term] * tf * (self.k1 + 1) / (tf + norm)
        return s

    def search(self, q_tokens: list[str], k: int = 10) -> list[str]:
        s = self.scores(q_tokens)
        order = np.lexsort((np.arange(self.N), -s))[:k]  # empates -> orden del corpus
        return [self.doc_ids[i] for i in order]

In [ ]:
corpus = carga_corpus(FAST_DEV_RUN)
docs, queries, qrels = corpus["docs"], corpus["queries"], corpus["qrels"]
doc_ids = [d["id"] for d in docs]
doc_txt = [d["text"] for d in docs]
q_txt = [q["text"] for q in queries]
print("fuente:", corpus["source"], "| docs:", dict(Counter(d["lang"] for d in docs)), "| queries:", dict(Counter(q["lang"] for q in queries)))


def evalua(S, k=10):
    """S: (n_queries, n_docs) similitudes -> métricas canónicas @k sobre todas las queries."""
    run = {q["qid"]: rank_from_scores(S[i], doc_ids, 100) for i, q in enumerate(queries)}
    return evaluate(run, qrels, k)

In [ ]:
# Partición determinista de las consultas: validación / test (≈ 50/50 por hash del qid; no estratificada)
import zlib
split = {q["qid"]: ("val" if zlib.crc32(q["qid"].encode()) % 2 == 0 else "test") for q in queries}
idx = {s: [i for i, q in enumerate(queries) if split[q["qid"]] == s] for s in ("val", "test")}


def evalua_split(S, s, k=10, lang=None):
    ii = [i for i in idx[s] if lang is None or queries[i]["lang"] == lang]
    run = {queries[i]["qid"]: rank_from_scores(S[i], doc_ids, 100) for i in ii}
    return evaluate(run, {queries[i]["qid"]: qrels[queries[i]["qid"]] for i in ii}, k)


print({s: len(v) for s, v in idx.items()})

In [ ]:
# ── Fallback documentado: si no hay acceso al hub, construimos dos encoders diminutos LOCALES con la API de HF ──
#    mini-BERT "crudo" (solo MLM, como un BERT sin ajustar) y mini-SBERT (el mismo + 200 pasos contrastivos con
#    pares débiles frase-1 ↔ resto-del-documento; el módulo 11 explica este entrenamiento paso a paso).
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
from transformers import BertConfig, BertForMaskedLM, BertModel, BertTokenizerFast, DataCollatorForLanguageModeling


def vocab_wordpiece_determinista(textos, vocab_size=2000):
    """Vocabulario con convención WordPiece ("##" = continuación de palabra), REPRODUCIBLE entre ejecuciones.
    El WordPiece trainer de `tokenizers` rompe empates según el orden de un HashMap de Rust, que cambia en cada proceso:
    el vocabulario (y con él todo lo que se entrena encima) variaba entre ejecuciones. Unigram sí es determinista."""
    uni = Tokenizer(models.Unigram())
    uni.normalizer = normalizers.BertNormalizer(lowercase=True, strip_accents=False)
    uni.pre_tokenizer = pre_tokenizers.Sequence([pre_tokenizers.BertPreTokenizer(), pre_tokenizers.Metaspace()])
    uni.train_from_iterator(textos, trainers.UnigramTrainer(vocab_size=vocab_size, unk_token="[UNK]",
                                                            special_tokens=["[UNK]"], show_progress=False))
    chars = sorted({c for t in textos for c in uni.normalizer.normalize_str(t) if not c.isspace()})
    piezas = [p for p, _ in sorted(uni.get_vocab().items(), key=lambda x: x[1]) if p not in ("[UNK]", "▁")]
    piezas = [p[1:] if p.startswith("▁") else "##" + p for p in piezas]       # "▁x" inicia palabra; "x" → "##x"
    vocab = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"] + chars + ["##" + c for c in chars] + piezas
    return list(dict.fromkeys(vocab))


def construye_fallback(textos, out="mini_encoders", steps_mlm=200, steps_cl=200, seed=SEED):
    os.makedirs(f"{out}/raw", exist_ok=True)
    with open(f"{out}/raw/vocab.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(vocab_wordpiece_determinista(textos)) + "\n")
    tk = BertTokenizerFast(f"{out}/raw/vocab.txt", do_lower_case=True, strip_accents=False)
    torch.manual_seed(seed); rng = random.Random(seed)
    mlm = BertForMaskedLM(BertConfig(vocab_size=len(tk), hidden_size=128, num_hidden_layers=2, num_attention_heads=4,
                                     intermediate_size=256, max_position_embeddings=128))
    coll = DataCollatorForLanguageModeling(tk, mlm_probability=0.15)
    feats = [{"input_ids": x} for x in tk(textos, truncation=True, max_length=64)["input_ids"]]
    opt = torch.optim.AdamW(mlm.parameters(), lr=1e-3)
    for _ in range(steps_mlm):
        loss = mlm(**coll([feats[rng.randrange(len(feats))] for _ in range(32)])).loss
        opt.zero_grad(); loss.backward(); opt.step()
    mlm.bert.save_pretrained(f"{out}/raw"); tk.save_pretrained(f"{out}/raw")
    # pares débiles (sin etiquetas): primera frase <-> resto del documento
    pares = []
    for t in textos:
        fr = re.split(r"(?<=[.!?])\s+", t.strip())
        if len(fr) >= 3:
            pares.append((fr[0], " ".join(fr[1:])))
    enc = copy.deepcopy(mlm.bert); opt = torch.optim.AdamW(enc.parameters(), lr=5e-4)
    def emb(b):  # media con máscara (autocontenida para no depender de otras celdas)
        m = b["attention_mask"].unsqueeze(-1).float()
        return F.normalize((enc(**b).last_hidden_state * m).sum(1) / m.sum(1), dim=-1)
    for _ in range(steps_cl):
        lote = rng.sample(pares, 32)
        a = tk([p[0] for p in lote], padding=True, truncation=True, max_length=64, return_tensors="pt")
        b = tk([p[1] for p in lote], padding=True, truncation=True, max_length=64, return_tensors="pt")
        loss = F.cross_entropy(emb(a) @ emb(b).T / 0.05, torch.arange(len(lote)))
        opt.zero_grad(); loss.backward(); opt.step()
    enc.save_pretrained(f"{out}/sbert"); tk.save_pretrained(f"{out}/sbert")
    return f"{out}/raw", f"{out}/sbert"

In [ ]:
from transformers import AutoModel, AutoTokenizer


class Encoder:
    """Envuelve cualquier encoder de HF: estados por token de cualquier capa + pooling elegido. prefijos = (query, doc)."""

    def __init__(self, nombre, ruta, prefijos=("", ""), max_len=128):
        self.nombre, self.ruta, self.prefijos, self.max_len = nombre, ruta, prefijos, max_len
        self.tok = AutoTokenizer.from_pretrained(ruta)
        self.model = AutoModel.from_pretrained(ruta).to(device).eval()

    @torch.no_grad()
    def estados(self, textos, capa=-1, bs=32):
        """Lista de (h (b,T,d) en CPU, mask (b,T)) por lotes."""
        out = []
        for i in range(0, len(textos), bs):
            b = self.tok(textos[i:i + bs], padding=True, truncation=True, max_length=self.max_len, return_tensors="pt").to(device)
            hs = self.model(**b, output_hidden_states=True).hidden_states
            out.append((hs[capa].float().cpu(), b["attention_mask"].cpu()))
        return out

    def encode(self, textos, pooling="mean", capa=-1, tipo="doc"):
        pre = self.prefijos[0] if tipo == "query" else self.prefijos[1]
        return torch.cat([POOLS[pooling](h, m) for h, m in self.estados([pre + t for t in textos], capa)]).numpy()


def carga_encoders(textos):
    """Modelos reales (si hay hub) o los mini-encoders de fallback. Devuelve {nombre: Encoder}."""
    reales = [("BERT multilingüe crudo", "distilbert-base-multilingual-cased", ("", "")),
              ("SBERT multilingüe", "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", ("", "")),
              ("E5 multilingüe small", "intfloat/multilingual-e5-small", ("query: ", "passage: "))]
    encs = {}
    for nombre, ruta, pre in reales:
        try:
            encs[nombre] = Encoder(nombre, ruta, pre)
        except Exception as e:
            print(f"   (sin {ruta}: {type(e).__name__})")
    if not encs:
        print("⚠️  Sin acceso al hub: construyo los mini-encoders de fallback (~1-2 min en CPU)")
        raw, sb = construye_fallback(textos)
        encs = {"mini-BERT crudo (fallback)": Encoder("mini-BERT crudo (fallback)", raw),
                "mini-SBERT (fallback)": Encoder("mini-SBERT (fallback)", sb)}
    return encs

In [ ]:
def coseno(Q, D):
    Qn = Q / np.linalg.norm(Q, axis=1, keepdims=True).clip(1e-12)
    Dn = D / np.linalg.norm(D, axis=1, keepdims=True).clip(1e-12)
    return Qn @ Dn.T


encs = carga_encoders(doc_txt)
langs_doc = np.array([d["lang"] for d in docs])
langs_q = np.array([q["lang"] for q in queries])

## TODO 1 · Pooling con máscara
Implementa las cuatro funciones: entrada `h` (B, T, d) y `mask` (B, T) con 1 = token real; salida (B, d).

<details><summary>🪜 Pista 1</summary>

Media: multiplica por la máscara expandida a (B, T, 1) y divide por el nº de tokens reales. Máximo: rellena las posiciones de padding con el mínimo representable antes de `max`.
</details>
<details><summary>🪜 Pista 2</summary>

Último token real: `T - 1 - mask.flip(1).float().argmax(1)` da el índice del último 1 de cada fila (funciona con padding a la izquierda y a la derecha). Luego `h[torch.arange(B), idx]`.
</details>

In [ ]:
def media_mascara(h, mask):
    # TODO 1a
    raise NotImplementedError


def max_mascara(h, mask):
    # TODO 1b
    raise NotImplementedError


def cls_pool(h, mask):
    return h[:, 0]


def ultimo_token(h, mask):
    # TODO 1c
    raise NotImplementedError


POOLS = {"cls": cls_pool, "mean": media_mascara, "max": max_mascara, "last": ultimo_token}

In [ ]:
# Tests de R1 (no los modifiques)
def tests_pooling():
    torch.manual_seed(0)
    h = torch.randn(2, 5, 3)
    m = torch.tensor([[1, 1, 1, 1, 1], [1, 1, 0, 0, 0]])
    assert torch.allclose(media_mascara(h, m)[1], h[1, :2].mean(0)), "media"
    assert torch.allclose(max_mascara(h, m)[1], h[1, :2].max(0).values), "máximo"
    assert torch.equal(ultimo_token(h, m)[1], h[1, 1]), "último (padding derecha)"
    m2 = torch.tensor([[1, 1, 1, 1, 1], [0, 0, 0, 1, 1]])
    assert torch.equal(ultimo_token(h, m2)[1], h[1, 4]), "último (padding izquierda)"
    return True


try:
    R1 = tests_pooling(); print("✅ R1 pooling")
except NotImplementedError:
    R1 = False; print("⏳ TODO 1 pendiente")

## TODO 2 · Whitening y TODO 3 · centrado por idioma
- `ajusta_whitening(X, k)` → `(mu, W)` con `W = U[:, :k] / sqrt(S[:k] + eps)` a partir de la SVD de la covarianza de `X`.
- `centra_por_idioma(X, langs, medias=None)` → `(X_centrado, medias)`: resta a cada fila la media de su idioma; si `medias` se pasa (calculadas con los **documentos**), úsalas (así se transforman las consultas).

<details><summary>🪜 Pista 1</summary>

¿Con qué datos ajustas `mu`, `W` y las medias por idioma? Solo con los **documentos** (el índice). Las consultas se transforman con esos parámetros.
</details>

In [ ]:
def ajusta_whitening(X, k=None, eps=1e-6):
    # TODO 2
    raise NotImplementedError


def centra_por_idioma(X, langs, medias=None):
    # TODO 3
    raise NotImplementedError

## TODO 4 · Rejilla de configuraciones en validación
Completa `postprocesa(D, Q, modo)` para los modos `"ninguno"`, `"centrado"`, `"whitening"` (con $k$ = mitad de la dimensión) y `"idioma"`. La celda siguiente recorre encoders × poolings × modos y guarda nDCG@10 de **validación**.

<details><summary>🪜 Pista 1</summary>

`"centrado"`: resta `D.mean(0)` a ambos. `"whitening"`: `mu, W = ajusta_whitening(D, D.shape[1] // 2)` y aplica `(X - mu) @ W` a ambos. `"idioma"`: `centra_por_idioma(D, langs_doc)` y luego las consultas con las medias de los documentos (`langs_q`; el código de las consultas `code` usa la media de los documentos `code`).
</details>

In [ ]:
MODOS = ["ninguno", "centrado", "whitening", "idioma"]


def postprocesa(D, Q, modo):
    # TODO 4
    raise NotImplementedError


def rejilla():
    filas, cache = [], {}
    for n, e in encs.items():
        for p in POOLS:
            D, Q = e.encode(doc_txt, p, tipo="doc"), e.encode(q_txt, p, tipo="query")
            cache[(n, p)] = (D, Q)
            for modo in MODOS:
                D2, Q2 = postprocesa(D, Q, modo)
                filas.append(dict(encoder=n, pooling=p, modo=modo, ndcg_val=evalua_split(coseno(Q2, D2), "val")["ndcg"]))
    return filas, cache


try:
    filas, cache = rejilla()
    print(f"{len(filas)} configuraciones evaluadas en validación")
except NotImplementedError:
    filas = None; print("⏳ TODO 1–4 pendientes")

## TODO 5 · Elegir en validación, reportar en test y rúbrica
La celda ya está escrita: elige la mejor configuración **en validación**, la mide en **test**, la compara con «media sin post-procesado» del mismo encoder y con BM25, desglosa por idioma y comprueba la rúbrica. Escribe después 4–5 frases con tu recomendación para Producto.

In [ ]:
def informe_y_rubrica(filas, cache):
    import pandas as pd
    df = pd.DataFrame(filas).sort_values("ndcg_val", ascending=False)
    best = df.iloc[0]
    D, Q = cache[(best.encoder, best.pooling)]
    D2, Q2 = postprocesa(D, Q, best.modo)
    S_best = coseno(Q2, D2)
    S_def = coseno(*cache[(best.encoder, "mean")][::-1])
    bm = BM25().fit([tokenize_simple(t) for t in doc_txt], doc_ids)
    S_bm = np.array([bm.scores(tokenize_simple(q)) for q in q_txt])
    t_best, t_def, t_bm = (evalua_split(S, "test")["ndcg"] for S in (S_best, S_def, S_bm))
    print(df.head(8).to_string(index=False))
    print(f"\nElegida en validación: {best.encoder} · {best.pooling} · {best.modo}")
    print(f"TEST nDCG@10: elegida={t_best:.3f} | media sin post-procesado={t_def:.3f} | BM25={t_bm:.3f}")
    for l in ("es", "en", "code"):
        print(f"   {l:5s} elegida={evalua_split(S_best, 'test', lang=l)['ndcg']:.3f} | BM25={evalua_split(S_bm, 'test', lang=l)['ndcg']:.3f}")
    mu, W = ajusta_whitening(D, D.shape[1] // 2)
    Xw = (D - mu) @ W
    R2 = np.abs(np.cov(Xw.T) - np.eye(Xw.shape[1])).max() < 1e-3
    Dl, med = centra_por_idioma(D, langs_doc)
    R3 = max(np.abs(Dl[langs_doc == l].mean(0)).max() for l in np.unique(langs_doc)) < 1e-6
    checks = {"R1 pooling": R1, "R2 whitening ≈ identidad": R2, "R3 centrado por idioma": R3,
              "R4 ≥ 32 configuraciones": len(df) >= 32, "R5 test ≥ default − 0.03": t_best >= t_def - 0.03, "R6 desglose impreso": True}
    for k_, ok in checks.items():
        print(("✅ " if ok else "❌ ") + k_)
    return dict(best=best.to_dict(), test_best=t_best, test_default=t_def, test_bm25=t_bm)


if filas:
    informe = informe_y_rubrica(filas, cache)

## Retos extra
1. **Elegir $k$ del whitening en validación** (8, 16, 32, 64, …) en vez de fijar $d/2$. ¿Cuánto cambia el resultado en test? ¿Y su varianza con otra semilla de partición?
2. **Intervalos de confianza**: bootstrap por consulta del nDCG@10 de test (módulo 15) para la elegida y la *default*. ¿Es significativa la diferencia?
3. **Híbrido rápido**: suma normalizada de scores densos y BM25 ($\alpha\in\{0.3, 0.5, 0.7\}$, elegido en validación). Es el adelanto del módulo 13.
4. **Coseno vs producto**: repite la mejor configuración con producto escalar sin normalizar.

## Reflexión
- ¿Por qué elegir en validación y reportar en test es imprescindible incluso con solo 32 configuraciones?
- ¿Qué parámetros de post-procesado tendrías que versionar junto al modelo en producción?
- Si BM25 gana en este benchmark de ítem conocido, ¿significa que la búsqueda semántica no sirve para Políglota? ¿Qué consultas faltan en el *golden set*?

---
## ⛔ SPOILER — intenta resolverlo primero
Solución de referencia completa: redefine las funciones y vuelve a ejecutar tests, rejilla, informe y rúbrica.

In [ ]:
def media_mascara(h, mask):
    m = mask.unsqueeze(-1).to(h.dtype)
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)


def max_mascara(h, mask):
    return h.masked_fill(mask.unsqueeze(-1) == 0, torch.finfo(h.dtype).min).max(1).values


def ultimo_token(h, mask):
    idx_ = mask.shape[1] - 1 - mask.flip(1).float().argmax(1)
    return h[torch.arange(h.shape[0]), idx_]


POOLS = {"cls": cls_pool, "mean": media_mascara, "max": max_mascara, "last": ultimo_token}


def ajusta_whitening(X, k=None, eps=1e-6):
    mu = X.mean(0, keepdims=True)
    U, S, _ = np.linalg.svd(np.cov((X - mu).T))
    k = k or X.shape[1]
    return mu, U[:, :k] / np.sqrt(S[:k] + eps)


def centra_por_idioma(X, langs, medias=None):
    medias = medias or {l: X[langs == l].mean(0) for l in np.unique(langs)}
    Xc = X.copy()
    for l, m in medias.items():
        Xc[langs == l] -= m
    return Xc, medias


def postprocesa(D, Q, modo):
    if modo == "ninguno":
        return D, Q
    if modo == "centrado":
        mu = D.mean(0, keepdims=True)
        return D - mu, Q - mu
    if modo == "whitening":
        mu, W = ajusta_whitening(D, D.shape[1] // 2)
        return (D - mu) @ W, (Q - mu) @ W
    if modo == "idioma":
        D2, med = centra_por_idioma(D, langs_doc)
        Q2, _ = centra_por_idioma(Q, langs_q, med)
        return D2, Q2
    raise ValueError(modo)


R1 = tests_pooling(); print("✅ R1 pooling")
filas, cache = rejilla()
informe = informe_y_rubrica(filas, cache)